#DummyClassifier

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

In [2]:
# 定位项目路径并创建输出目录
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Cannot locate project root. Please run this notebook inside the project folder or notebook folder."
    )

DATA_DIR = PROJECT_DIR / "data" / "raw"

OUTPUT_DIR = PROJECT_DIR / "reports" / "model" / "baseline" / "dummy"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_DIR: d:\A_projects\kaggle-customer-churn-prediction
DATA_DIR: d:\A_projects\kaggle-customer-churn-prediction\data\raw
OUTPUT_DIR: d:\A_projects\kaggle-customer-churn-prediction\reports\model\baseline\dummy


In [3]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_submission_path = DATA_DIR / "sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample submission shape:", sample_submission.shape)

display(train.head())

train shape: (594194, 21)
test shape: (254655, 20)
sample submission shape: (254655, 2)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [4]:
#划分特征与目标变量
TARGET_COL = "Churn"

if TARGET_COL not in train.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in train.csv")

id_candidates = {
    "id",
    "customerid",
    "customer_id",
    "clientid",
    "client_id",
    "userid",
    "user_id",
}

id_cols = [col for col in train.columns if col.lower() in id_candidates]

feature_cols = [col for col in train.columns if col not in id_cols + [TARGET_COL]]

X = train[feature_cols]
y = train[TARGET_COL]

print("ID columns:", id_cols)
print("Target column:", TARGET_COL)
print("Number of features:", len(feature_cols))
print("Target distribution:")
display(y.value_counts(normalize=True).rename("proportion").to_frame())

ID columns: ['id']
Target column: Churn
Number of features: 19
Target distribution:


,proportion
Churn,
No,0.774792
Yes,0.225208


In [5]:
#划分训练集与验证集
RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)

print("Train target distribution:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())

print("Valid target distribution:")
display(y_valid.value_counts(normalize=True).rename("proportion").to_frame())

X_train shape: (475355, 19)
X_valid shape: (118839, 19)
Train target distribution:


,proportion
Churn,
No,0.774791
Yes,0.225209


Valid target distribution:


,proportion
Churn,
No,0.774796
Yes,0.225204


In [6]:
#训练基线模型并评估性能
dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=RANDOM_STATE,
)

dummy_model.fit(X_train, y_train)

y_valid_pred = dummy_model.predict(X_valid)

print("Dummy model classes:", dummy_model.classes_)
print("Predicted class distribution:")
display(pd.Series(y_valid_pred).value_counts(normalize=True).rename("proportion").to_frame())

Dummy model classes: ['No' 'Yes']
Predicted class distribution:


,proportion
No,1.0


In [7]:
#计算验证集指标
classes = dummy_model.classes_

if len(classes) != 2:
    raise ValueError("This notebook assumes a binary classification task.")

# 自动识别正类
if 1 in classes:
    positive_class = 1
elif "Yes" in classes:
    positive_class = "Yes"
elif "yes" in classes:
    positive_class = "yes"
elif "True" in classes:
    positive_class = "True"
else:
    positive_class = classes[1]

positive_index = np.where(classes == positive_class)[0][0]

y_valid_binary = (y_valid == positive_class).astype(int)

if hasattr(dummy_model, "predict_proba"):
    y_valid_proba = dummy_model.predict_proba(X_valid)[:, positive_index]
else:
    y_valid_proba = None

metrics = {
    "model": "DummyClassifier",
    "strategy": "most_frequent",
    "positive_class": positive_class,
    "train_size": len(X_train),
    "valid_size": len(X_valid),
    "accuracy": accuracy_score(y_valid, y_valid_pred),
    "balanced_accuracy": balanced_accuracy_score(y_valid, y_valid_pred),
    "precision": precision_score(y_valid, y_valid_pred, pos_label=positive_class, zero_division=0),
    "recall": recall_score(y_valid, y_valid_pred, pos_label=positive_class, zero_division=0),
    "f1": f1_score(y_valid, y_valid_pred, pos_label=positive_class, zero_division=0),
}

if y_valid_proba is not None:
    metrics["roc_auc"] = roc_auc_score(y_valid_binary, y_valid_proba)
    metrics["average_precision"] = average_precision_score(y_valid_binary, y_valid_proba)

metrics_df = pd.DataFrame([metrics])

display(metrics_df)

,model,strategy,positive_class,train_size,valid_size,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,DummyClassifier,most_frequent,Yes,475355,118839,0.774796,0.5,0.0,0.0,0.0,0.5,0.225204


In [8]:
#保存分类报告和混淆矩阵
report_dict = classification_report(
    y_valid,
    y_valid_pred,
    output_dict=True,
    zero_division=0,
)

classification_report_df = pd.DataFrame(report_dict).T

cm = confusion_matrix(y_valid, y_valid_pred, labels=classes)
confusion_matrix_df = pd.DataFrame(
    cm,
    index=[f"actual_{cls}" for cls in classes],
    columns=[f"pred_{cls}" for cls in classes],
)

metrics_df.to_csv(TABLE_DIR / "dummy_metrics.csv", index=False)
classification_report_df.to_csv(TABLE_DIR / "dummy_classification_report.csv")
confusion_matrix_df.to_csv(TABLE_DIR / "dummy_confusion_matrix.csv")

display(classification_report_df)
display(confusion_matrix_df)

print("Saved tables to:", TABLE_DIR)

,precision,recall,f1-score,support
No,0.774796,1.000000,0.873110,92076.000000
Yes,0.000000,0.000000,0.000000,26763.000000
accuracy,0.774796,0.774796,0.774796,0.774796
macro avg,0.387398,0.500000,0.436555,118839.000000
weighted avg,0.600309,0.774796,0.676482,118839.000000


,pred_No,pred_Yes
actual_No,92076,0
actual_Yes,26763,0


Saved tables to: d:\A_projects\kaggle-customer-churn-prediction\reports\model\baseline\dummy\tables
